# Complete Transit Nodes Processing Pipeline

This notebook integrates the complete workflow:

## Part 1: H3 Hexagon Processing (Steps 1-4)
1. Load transit nodes from CSV
2. Assign H3 hexagonal indices
3. Group hexagons by 120m proximity (edge-to-edge)
4. Geocode addresses (optional)
5. Export grouped hexagons

## Part 2: Demand Data Processing (Steps 5-8)
5. Load grouped hexagons
6. Tag with spatial information (metro areas, districts)
7. Add daily demand from regional transport models
8. Create aggregated hub groups with demand statistics

## Part 3: Influence Area Processing (Steps 9-11) [Optional]
9. Create buffer zones (0-600m, 600-1000m, 1000-1200m)
10. Calculate population and employment from TAZ data
11. Identify proximity to bus terminals

## Part 4: Final Export and Analysis (Step 12)
12. Export complete dataset with analysis

**Total Runtime:** ~8-10 minutes for complete pipeline
- Part 1: ~2 minutes (without geocoding)
- Part 2: ~3 minutes
- Part 3: ~2-3 minutes
- Part 4: ~1 minute

---
# PART 1: H3 HEXAGON PROCESSING
---

## Step 1.1: Setup and Configuration

In [ ]:
# Install required packages (uncomment if needed)
# !pip install h3 geopandas pandas shapely geopy openpyxl

In [ ]:
import h3
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
from shapely import wkt
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import warnings
import os
warnings.filterwarnings('ignore')

print("✓ All libraries loaded successfully!")

## Step 1.2: Configure File Paths for Part 1

In [ ]:
# ============================================================================
# PART 1 CONFIGURATION - H3 HEXAGON PROCESSING
# ============================================================================

# Input: Transit nodes CSV
INPUT_NODES_CSV = '/path/to/All_nodes+lines.csv'

# Input: Lines and Planned Mode CSV (for mode determination)
LINES_MODE_CSV = '/path/to/Lines_and_Planned_Mode.csv'

# Output: H3 hexagons with groups
OUTPUT_H3_HEXAGONS = '/path/to/output/transit_h3_hexagons.csv'

# Processing parameters
H3_RESOLUTION = 10        # H3 resolution (10 = ~15m hexagons)
BUFFER_DISTANCE = 120     # Buffer distance in meters for grouping (edge-to-edge)
SKIP_GEOCODING = True     # Set to False to geocode addresses (much slower)

# Coordinate reference systems
CRS_PROJECTED = "EPSG:2039"  # Israel TM Grid (for meter-based operations)
CRS_WGS84 = "EPSG:4326"      # WGS84 (for H3 and geocoding)

print("Part 1 Configuration:")
print(f"  H3 Resolution: {H3_RESOLUTION}")
print(f"  Buffer Distance: {BUFFER_DISTANCE}m (edge-to-edge)")
print(f"  Skip Geocoding: {SKIP_GEOCODING}")
print(f"  Input: {INPUT_NODES_CSV}")
print(f"  Output: {OUTPUT_H3_HEXAGONS}")

## Step 1.3: Load Transit Nodes Data

In [ ]:
print("Loading transit nodes data...")

# Read CSV with pandas
df = pd.read_csv(INPUT_NODES_CSV, encoding='windows-1255')

# Check if geometry column exists
if 'geometry' in df.columns:
    # If geometry exists as WKT string, convert it
    df['geometry'] = df['geometry'].apply(lambda x: wkt.loads(x) if pd.notna(x) else None)
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=CRS_PROJECTED)
elif 'X' in df.columns and 'Y' in df.columns:
    # Create geometry from X, Y coordinates
    geometry = [Point(x, y) for x, y in zip(df['X'], df['Y'])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=CRS_PROJECTED)
else:
    raise ValueError("CSV must contain either 'geometry' column or both 'X' and 'Y' columns")

print(f"✓ Loaded {len(gdf)} records")
print(f"  CRS: {gdf.crs}")
print(f"  Columns: {list(gdf.columns)}")
print(f"\nFirst 3 rows:")
gdf.head(3)

## Step 1.4: Assign H3 Indices and Aggregate Lines

In [ ]:
print("Step 1.4: Loading Lines and Planned Mode data, then assigning H3 indices...")

# ============================================================================
# IMPORTANT: Configure path to Lines_and_Planned_Mode.csv
# ============================================================================
LINES_MODE_CSV = '/path/to/Lines_and_Planned_Mode.csv'  # UPDATE THIS PATH!

print(f"  Loading mode data from: {LINES_MODE_CSV}")
lines_mode = pd.read_csv(LINES_MODE_CSV, encoding='windows-1255')
print(f"  ✓ Loaded {len(lines_mode)} line records")
print(f"  Columns: {list(lines_mode.columns)}")

# Convert to WGS84 for H3
gdf_wgs84 = gdf.to_crs(CRS_WGS84)

# Extract lat/lon
gdf_wgs84['lat'] = gdf_wgs84.geometry.y
gdf_wgs84['lon'] = gdf_wgs84.geometry.x

# Assign H3 index
gdf_wgs84['h3_index'] = gdf_wgs84.apply(
    lambda row: h3.latlng_to_cell(row['lat'], row['lon'], H3_RESOLUTION),
    axis=1
)

print(f"  ✓ Assigned H3 indices to {len(gdf_wgs84)} records")
print(f"  Unique H3 hexagons (before filtering): {gdf_wgs84['h3_index'].nunique()}")

# ============================================================================
# Filter out old Metronit lines (Haifa area, starting with 'm')
# ============================================================================
print("\n  Filtering out old Metronit lines...")
remove_old_metronit = []
if 'Area' in lines_mode.columns:
    haifa_lines = lines_mode[lines_mode['Area'] == 'Haifa']['Line_ModelName'].tolist()
    for line in haifa_lines:
        if str(line)[:1] == 'm':
            remove_old_metronit.append(line)
            
    if remove_old_metronit:
        print(f"    Removing old Metronit: {', '.join(map(str, remove_old_metronit))}")
        gdf_wgs84 = gdf_wgs84[~gdf_wgs84['LINE_ID'].isin(remove_old_metronit)]
        print(f"    ✓ After filtering: {len(gdf_wgs84)} records")
    else:
        print(f"    No old Metronit lines found")
else:
    print(f"    ⚠ 'Area' column not found in Lines_and_Planned_Mode.csv, skipping Metronit filter")

# ============================================================================
# Filter out Netanya LRT151/152 lines
# ============================================================================
print("\n  Filtering out Netanya LRT151/152 lines...")
remove_netanya15 = []
if 'Area' in lines_mode.columns:
    netanya_lines = lines_mode[lines_mode['Area'] == 'Netanya']['Line_ModelName'].tolist()
    for line in netanya_lines:
        if line == 'LRT151' or line == 'LRT152':
            remove_netanya15.append(line)
            
    if remove_netanya15:
        print(f"    Removing Netanya lines: {', '.join(map(str, remove_netanya15))}")
        gdf_wgs84 = gdf_wgs84[~gdf_wgs84['LINE_ID'].isin(remove_netanya15)]
        print(f"    ✓ After filtering: {len(gdf_wgs84)} records")
    else:
        print(f"    No Netanya LRT151/152 lines found")
else:
    print(f"    ⚠ 'Area' column not found, skipping Netanya filter")

# ============================================================================
# Merge with Lines and Planned Mode to get Mode_Planned
# ============================================================================
print("\n  Merging with mode data...")
gdf_with_mode = gdf_wgs84.merge(
    lines_mode, 
    left_on='LINE_ID', 
    right_on='Line_ModelName', 
    how='left',
    validate='many_to_many'
)

print(f"  ✓ Merged data: {len(gdf_with_mode)} records")

# Check if Mode_Planned column exists after merge
if 'Mode_Planned' not in gdf_with_mode.columns:
    print("  ⚠ WARNING: 'Mode_Planned' column not found after merge!")
    print(f"  Available columns: {list(gdf_with_mode.columns)}")
    print("  Using 'Bus' as default mode for all lines")
    gdf_with_mode['Mode_Planned'] = 'Bus'

# ============================================================================
# Group by h3_index, node, and Mode_Planned, then aggregate lines
# ============================================================================
print("\n  Aggregating by H3 hexagon and mode...")

h3_grouped = gdf_with_mode.groupby(['h3_index', 'node', 'Mode_Planned']).agg({
    'Line_ModelName': ['nunique', lambda x: list(x.unique())]
}).reset_index()

# Flatten column names
h3_grouped.columns = ['h3_index', 'node', 'Mode_Planned', 'Line_Nunique', 'Line_Unique']

print(f"  ✓ Created {len(h3_grouped)} unique H3-node-mode combinations")

# ============================================================================
# Further aggregate by h3_index only (combining all modes)
# ============================================================================
print("\n  Final aggregation by H3 hexagon...")

h3_final = h3_grouped.groupby('h3_index').agg({
    'node': 'first',
    'Mode_Planned': lambda x: list(x.unique()),  # List of all modes
    'Line_Nunique': 'sum',  # Total unique lines across all modes
    'Line_Unique': lambda x: list(set([item for sublist in x for item in sublist]))  # Flatten and deduplicate lines
}).reset_index()

print(f"  ✓ Aggregated to {len(h3_final)} unique hexagons")

# ============================================================================
# Create geometry from H3 indices
# ============================================================================
def h3_to_polygon(h3_index):
    """Convert H3 index to Shapely Polygon."""
    boundary = h3.cell_to_boundary(h3_index)
    # H3 returns (lat, lon), need (lon, lat) for Shapely
    return Polygon([(lon, lat) for lat, lon in boundary])

print("  Creating hexagon geometries...")
h3_final['geometry'] = h3_final['h3_index'].apply(h3_to_polygon)

# Convert to GeoDataFrame
gdf_h3 = gpd.GeoDataFrame(h3_final, geometry='geometry', crs=CRS_WGS84)

print(f"\n✓ Step 1.4 complete! Created {len(gdf_h3)} hexagons")
print(f"  Modes distribution:")
if 'Mode_Planned' in gdf_h3.columns:
    all_modes = [mode for modes_list in gdf_h3['Mode_Planned'] for mode in modes_list]
    mode_counts = pd.Series(all_modes).value_counts()
    for mode, count in mode_counts.items():
        print(f"    {mode}: {count}")

gdf_h3.head(3)

## Step 1.5: Create Groups Based on 120m Buffer (Edge-to-Edge)

In [ ]:
print(f"Step 1.5: Creating groups based on {BUFFER_DISTANCE}m buffer...")
print("Note: Measuring edge-to-edge distance (border to border)")
print("Note: Using transitive grouping (A near B, B near C → all same group)")

# Convert to projected CRS for meter-based operations
gdf_proj = gdf_h3.to_crs(CRS_PROJECTED).copy()
gdf_proj = gdf_proj.reset_index(drop=True)

n = len(gdf_proj)
print(f"  Processing {n} hexagons...")

In [ ]:
# Build spatial index for efficient querying
print("  Building spatial index...")
sindex = gdf_proj.sindex

# Union-Find data structure
parent = list(range(n))

def find(x):
    """Find root with path compression."""
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    """Union two components."""
    root_x = find(x)
    root_y = find(y)
    if root_x != root_y:
        parent[root_y] = root_x

print("  Union-Find structure initialized")

In [ ]:
# Find all pairs of hexagons within buffer_distance (edge to edge)
print("  Finding neighbors within buffer distance (edge-to-edge)...")
pairs_found = 0

# Add small tolerance for touching hexagons (floating point precision)
tolerance = 0.1  # 10cm tolerance for "touching"
effective_distance = BUFFER_DISTANCE + tolerance

for idx1 in range(n):
    geom1 = gdf_proj.iloc[idx1].geometry
    
    # Query spatial index for potential neighbors
    minx, miny, maxx, maxy = geom1.bounds
    search_bounds = (
        minx - BUFFER_DISTANCE,
        miny - BUFFER_DISTANCE,
        maxx + BUFFER_DISTANCE,
        maxy + BUFFER_DISTANCE
    )
    
    possible_neighbors = list(sindex.intersection(search_bounds))
    
    # Check actual edge-to-edge distance
    for idx2 in possible_neighbors:
        if idx2 <= idx1:  # Avoid checking pairs twice
            continue
        
        geom2 = gdf_proj.iloc[idx2].geometry
        distance = geom1.distance(geom2)
        
        if distance <= effective_distance:
            union(idx1, idx2)
            pairs_found += 1
    
    if (idx1 + 1) % 100 == 0:
        print(f"    Processed {idx1 + 1}/{n} hexagons...")

print(f"  ✓ Found {pairs_found} neighbor pairs within {BUFFER_DISTANCE}m (edge-to-edge)")

In [ ]:
# Assign group labels
print("  Assigning group IDs...")
root_to_group = {}
group_counter = 0

for idx in range(n):
    root = find(idx)
    if root not in root_to_group:
        root_to_group[root] = group_counter
        group_counter += 1

labels = [root_to_group[find(idx)] for idx in range(n)]
gdf_h3['group'] = labels
n_groups = len(root_to_group)

# Statistics
group_sizes = gdf_h3.groupby('group').size()

print(f"\n✓ Step 1.5 complete!")
print(f"  Created {n_groups} groups from {len(gdf_h3)} hexagons")
print(f"  Single hexagon groups: {(group_sizes == 1).sum()}")
print(f"  Multi-hexagon groups: {(group_sizes > 1).sum()}")
print(f"  Largest group: {group_sizes.max()} hexagons")
print(f"  Average group size: {group_sizes.mean():.2f} hexagons")

## Step 1.6: Geocode Addresses (Optional)

In [ ]:
if not SKIP_GEOCODING:
    print("Step 1.6: Geocoding addresses...")
    print("Note: This step uses Nominatim and may take several minutes.")
    print("      Rate limited to 1 request per second.\n")
    
    # Initialize geocoder
    geolocator = Nominatim(user_agent='transit_h3_processor')
    geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1)
    
    # Get centroids in WGS84
    gdf_wgs84_final = gdf_h3.to_crs(CRS_WGS84)
    centroids = gdf_wgs84_final.geometry.centroid
    
    addresses = []
    total = len(centroids)
    
    for idx, centroid in enumerate(centroids):
        try:
            location = geocode(f"{centroid.y}, {centroid.x}", language='he')
            address = location.address if location else "Address not found"
            addresses.append(address)
            
            if (idx + 1) % 10 == 0:
                print(f"  Geocoded {idx + 1}/{total} addresses ({(idx+1)/total*100:.1f}%)")
        except Exception as e:
            print(f"  Error geocoding index {idx}: {e}")
            addresses.append("Geocoding error")
    
    gdf_h3['address'] = addresses
    print(f"\n✓ Step 1.6 complete! Geocoded {len(addresses)} locations")
    
else:
    print("Step 1.6: Skipping geocoding (as configured)")
    gdf_h3['address'] = 'Not geocoded'
    print("✓ Step 1.6 complete (skipped)")

## Step 1.7: Export H3 Hexagons with Groups

In [ ]:
print(f"Step 1.7: Exporting H3 hexagons to {OUTPUT_H3_HEXAGONS}...")

# Select final columns
output_columns = [
    'h3_index', 'node', 'Mode_Planned', 'Line_Nunique', 
    'Line_Unique', 'geometry', 'group', 'address'
]

gdf_output_part1 = gdf_h3[output_columns].copy()

# CRITICAL: Clean node column to ensure it's a single integer value
# This prevents '[np.int64(123)]' format in CSV that breaks Part 2
print("  Cleaning node column...")
if 'node' in gdf_output_part1.columns:
    # If node is somehow a list, take the first value
    gdf_output_part1['node'] = gdf_output_part1['node'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x
    )
    # Convert numpy types to Python int
    gdf_output_part1['node'] = gdf_output_part1['node'].apply(
        lambda x: int(x) if pd.notna(x) else x
    )
    print(f"  ✓ Node column cleaned: {gdf_output_part1['node'].dtype}")

# Ensure output directory exists
os.makedirs(os.path.dirname(OUTPUT_H3_HEXAGONS) if os.path.dirname(OUTPUT_H3_HEXAGONS) else '.', exist_ok=True)

# Export
if OUTPUT_H3_HEXAGONS.endswith('.csv'):
    gdf_output_part1['geometry'] = gdf_output_part1['geometry'].apply(lambda x: x.wkt)
    gdf_output_part1.to_csv(OUTPUT_H3_HEXAGONS, index=False, encoding='utf-8-sig')
elif OUTPUT_H3_HEXAGONS.endswith('.geojson'):
    gdf_output_part1.to_file(OUTPUT_H3_HEXAGONS, driver='GeoJSON')
elif OUTPUT_H3_HEXAGONS.endswith('.gpkg'):
    gdf_output_part1.to_file(OUTPUT_H3_HEXAGONS, driver='GPKG')
else:
    gdf_output_part1.to_file(OUTPUT_H3_HEXAGONS + '.geojson', driver='GeoJSON')

print(f"\n✓ Step 1.7 complete! File saved successfully.")
print(f"\n" + "="*80)
print("PART 1 COMPLETE - H3 HEXAGON PROCESSING")
print("="*80)
print(f"\nOutput: {OUTPUT_H3_HEXAGONS}")
print(f"Records: {len(gdf_output_part1)} INDIVIDUAL hexagons")
print(f"Groups: {gdf_output_part1['group'].nunique()} proximity groups identified")
print(f"\nColumns: {list(gdf_output_part1.columns)}")
print(f"\nNote: Grouping will happen in Part 2 AFTER demand assignment")
print(f"Ready for Part 2: Demand Data Processing")

---
# PART 2: DEMAND DATA PROCESSING
---

This part adds daily demand (boardings, alightings, transfers) to each hub group.

**What it does:**
1. Tags hubs with geographic area (from metro/districts shapefiles)
2. Loads demand data from Excel file with regional model sheets
3. Matches demand to nodes based on area (each region uses a different model)
4. Handles overlay models (Hadera, Haifa Metronit) that add to existing demand

**Regional Models:**
- Haifa (sheet: 5040_Daily)
- Tel Aviv (sheet: Daily_5087)
- Beer Sheva (sheet: Daily_BS)
- Jerusalem (sheet: Daily_Jerusalem)
- Ashdod-Ashkelon (sheet: Daily_5093)
- Hadera (overlay model)
- Haifa Metronit (overlay model)
- Rail (national)

**Required Files:**
- Metro shapefile with METRO_NAME column
- Districts shapefile with MACHOZ column
- Demand Excel file with regional sheets

## Step 2.1: Configure Paths for Part 2

In [ ]:
# ============================================================================
# PART 2 CONFIGURATION - DEMAND DATA PROCESSING
# ============================================================================

# Input: H3 hexagons from Part 1 (automatically set)
INPUT_H3_FOR_DEMAND = OUTPUT_H3_HEXAGONS

# Input: Demand data Excel file (with regional model sheets)
DEMAND_EXCEL = '/path/to/Nodes_w_results.xlsx'  # UPDATE THIS PATH!

# Input: Spatial layers for area tagging
METRO_SHP = '/path/to/metro.shp'           # Metro areas (with METRO_NAME column)
DISTRICTS_SHP = '/path/to/districts.shp'   # Districts (with MACHOZ column)

# Output: Hubs with demand
OUTPUT_HUBS_WITH_DEMAND = '/path/to/output/hubs_with_demand.csv'
OUTPUT_GROUPED_HUBS = '/path/to/output/Grouped_Hubs_Final.csv'

# Sheet name mapping: Excel sheet names -> region names
SHEET_NAME_MAPPING = {
    '5040_Daily': 'Haifa',
    'Daily_5087': 'Tel Aviv',
    'Daily_BS': 'Beer Sheva',
    'Daily_Hadera': 'Hadera',
    'Daily_Jerusalem': 'Jerusalem',
    'HaifaNewMetronit': 'Haifa Metronit',
    'Daily_5093': 'Ashdod-Ashkelon',
    'National': 'Rail',
    # Also accept already-mapped names
    'Haifa': 'Haifa',
    'TelAviv': 'Tel Aviv',
    'Tel Aviv': 'Tel Aviv',
    'BeerSheva': 'Beer Sheva',
    'Beer Sheva': 'Beer Sheva',
    'Hadera': 'Hadera',
    'Jerusalem': 'Jerusalem',
    'Ashdod': 'Ashdod-Ashkelon',
    'Ashdod-Ashkelon': 'Ashdod-Ashkelon',
    'Ashkelon': 'Ashdod-Ashkelon',
    'HaifaMetronit': 'Haifa Metronit',
    'Haifa Metronit': 'Haifa Metronit',
}

# Area to region mapping (Hebrew area names from spatial layers -> demand model regions)
AREA_TO_REGION = {
    'חיפה': 'Haifa',
    'צפון': 'Haifa',
    'תל אביב': 'Tel Aviv',
    'תל-אביב': 'Tel Aviv',
    'מרכז': 'Tel Aviv',
    'באר שבע': 'Beer Sheva',
    'דרום': 'Beer Sheva',
    'ירושלים': 'Jerusalem',
    'אשדוד': 'Ashdod-Ashkelon',
    'אשקלון': 'Ashdod-Ashkelon',
    # English names
    'Haifa': 'Haifa',
    'North': 'Haifa',
    'Tel Aviv': 'Tel Aviv',
    'Center': 'Tel Aviv',
    'Beer Sheva': 'Beer Sheva',
    'South': 'Beer Sheva',
    'Jerusalem': 'Jerusalem',
}

print("Part 2 Configuration:")
print(f"  Input H3: {INPUT_H3_FOR_DEMAND}")
print(f"  Demand Excel: {DEMAND_EXCEL}")
print(f"  Metro Shapefile: {METRO_SHP}")
print(f"  Districts Shapefile: {DISTRICTS_SHP}")
print(f"  Output (ungrouped): {OUTPUT_HUBS_WITH_DEMAND}")
print(f"  Output (grouped): {OUTPUT_GROUPED_HUBS}")
print(f"\nSheet name mappings: {len(SHEET_NAME_MAPPING)} entries")
print(f"Area to region mappings: {len(AREA_TO_REGION)} entries")

<cell_type>markdown</cell_type>## Step 2.2: Load H3 Hexagons from Part 1

In [ ]:
print("Step 2.2: Loading H3 hexagons from Part 1...")

# Check if file exists
if not os.path.exists(INPUT_H3_FOR_DEMAND):
    print(f"✗ Input file not found: {INPUT_H3_FOR_DEMAND}")
    print("  Make sure Part 1 completed successfully and the path is correct.")
    PART2_AVAILABLE = False
else:
    # Load H3 hexagons
    df_h3 = pd.read_csv(INPUT_H3_FOR_DEMAND, encoding='utf-8-sig')
    
    # Parse geometry from WKT
    if 'geometry' in df_h3.columns and df_h3['geometry'].dtype == 'object':
        df_h3['geometry'] = df_h3['geometry'].apply(lambda x: wkt.loads(x) if pd.notna(x) else None)
    
    gdf_demand = gpd.GeoDataFrame(df_h3, geometry='geometry', crs=CRS_WGS84)
    
    print(f"✓ Loaded {len(gdf_demand)} H3 hexagons")
    print(f"  Columns: {list(gdf_demand.columns)}")
    print(f"  Groups: {gdf_demand['group'].nunique()}")
    
    # Initialize demand columns
    gdf_demand['TotalDemand'] = 0.0
    gdf_demand['TotalTransfers'] = 0.0
    gdf_demand['area'] = None
    
    PART2_AVAILABLE = True
    
    gdf_demand.head(3)

<cell_type>markdown</cell_type>## Step 2.3: Tag Hubs with Area from Spatial Layers

This step assigns each hub to a geographic area using spatial joins:
1. First try metro shapefile (METRO_NAME column)
2. Fall back to districts shapefile (MACHOZ column)

The area determines which regional demand model to use for matching.

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.3: Tagging hubs with area from spatial layers...")
    
    metro_gdf = None
    districts_gdf = None
    
    # Load metro shapefile
    if os.path.exists(METRO_SHP):
        print(f"  Loading metro shapefile: {METRO_SHP}")
        metro_gdf = gpd.read_file(METRO_SHP)
        print(f"    ✓ Loaded {len(metro_gdf)} metro features")
        print(f"    Columns: {list(metro_gdf.columns)}")
    else:
        print(f"  ⚠ Metro shapefile not found: {METRO_SHP}")
    
    # Load districts shapefile
    if os.path.exists(DISTRICTS_SHP):
        print(f"  Loading districts shapefile: {DISTRICTS_SHP}")
        districts_gdf = gpd.read_file(DISTRICTS_SHP)
        print(f"    ✓ Loaded {len(districts_gdf)} district features")
        print(f"    Columns: {list(districts_gdf.columns)}")
    else:
        print(f"  ⚠ Districts shapefile not found: {DISTRICTS_SHP}")
    
    # Tag with metro layer first
    if metro_gdf is not None:
        try:
            print("\n  Tagging with metro layer...")
            hubs_wgs = gdf_demand.to_crs('EPSG:4326')
            metro_wgs = metro_gdf.to_crs('EPSG:4326')
            
            # Find metro name column
            metro_col = None
            for col in ['METRO_NAME', 'MetroName', 'metro_name', 'NAME', 'name', 'SHEM']:
                if col in metro_wgs.columns:
                    metro_col = col
                    break
            
            if metro_col:
                print(f"    Using column: {metro_col}")
                print(f"    Metro values: {metro_wgs[metro_col].unique().tolist()}")
                
                joined = gpd.sjoin(hubs_wgs, metro_wgs[[metro_col, 'geometry']], 
                                  how='left', predicate='intersects')
                if joined.index.duplicated().any():
                    joined = joined[~joined.index.duplicated(keep='first')]
                
                gdf_demand['area'] = joined[metro_col].values
                n_tagged = gdf_demand['area'].notna().sum()
                print(f"    ✓ Tagged {n_tagged}/{len(gdf_demand)} hubs")
            else:
                print(f"    ⚠ No metro name column found")
        except Exception as e:
            print(f"    ✗ Metro tagging failed: {e}")
    
    # Fall back to districts for untagged hubs
    if districts_gdf is not None:
        try:
            nan_mask = gdf_demand['area'].isna()
            if nan_mask.any():
                print(f"\n  Tagging {nan_mask.sum()} untagged hubs with districts layer...")
                hubs_wgs = gdf_demand[nan_mask].to_crs('EPSG:4326')
                districts_wgs = districts_gdf.to_crs('EPSG:4326')
                
                # Find district name column
                district_col = None
                for col in ['MACHOZ', 'SHEM_MACHOZ', 'SHEM_NAFA', 'District', 'NAME', 'SHEM']:
                    if col in districts_wgs.columns:
                        district_col = col
                        break
                
                if district_col:
                    print(f"    Using column: {district_col}")
                    print(f"    District values: {districts_wgs[district_col].unique().tolist()}")
                    
                    joined = gpd.sjoin(hubs_wgs, districts_wgs[[district_col, 'geometry']], 
                                      how='left', predicate='within')
                    if joined.index.duplicated().any():
                        joined = joined[~joined.index.duplicated(keep='first')]
                    
                    # Update only NaN rows
                    for idx in joined.index:
                        if pd.notna(joined.loc[idx, district_col]):
                            gdf_demand.loc[idx, 'area'] = joined.loc[idx, district_col]
                    
                    n_tagged = gdf_demand['area'].notna().sum()
                    print(f"    ✓ Total tagged: {n_tagged}/{len(gdf_demand)} hubs")
                else:
                    print(f"    ⚠ No district column found")
        except Exception as e:
            print(f"    ✗ District tagging failed: {e}")
    
    # Fill remaining with 'Unknown'
    gdf_demand['area'] = gdf_demand['area'].fillna('Unknown')
    
    # Summary
    print("\n  Area distribution:")
    area_counts = gdf_demand['area'].value_counts()
    for area, count in area_counts.items():
        print(f"    {area}: {count}")
    
    print(f"\n✓ Step 2.3 complete!")
else:
    print("Step 2.3: Skipped (Part 2 not available)")

## Step 2.4: Per-Sheet Column Configuration

Different regional models use different column names for node IDs, boardings, and alightings.
This configuration maps each region to its specific column names.

In [ ]:
# Per-sheet column configurations
# Each entry maps a region to its column naming conventions
SHEET_COLUMN_CONFIG = {
    'Beer Sheva': {
        'node_cols': ['NODE_ID', 'Node', 'node'],
        'boardings_cols': ['Boardings', 'Boardings_Daily', 'TotalBoardings'],
        'alightings_cols': ['Alightings', 'Alightings_Daily', 'TotalAlight'],
        'transfer_cols': [],
    },
    'Hadera': {
        'node_cols': ['NodeID', 'Node', 'node'],
        'boardings_cols': ['On', 'Boardings', 'Boardings_Daily'],
        'alightings_cols': ['Off', 'Alightings', 'Alightings_Daily'],
        'transfer_cols': [],
    },
    'Tel Aviv': {
        'node_cols': ['Node', 'node'],
        'boardings_cols': ['TotalBoardings', 'Boardings', 'Boardings_Daily'],
        'alightings_cols': ['TotalAlight', 'Alightings', 'Alightings_Daily'],
        'transfer_cols': ['TransferBoardings', 'TransferAlight'],
    },
    'Ashdod-Ashkelon': {
        'node_cols': ['Node', 'node'],
        'boardings_cols': ['InitialBoardings', 'Boardings', 'TotalBoardings'],
        'alightings_cols': ['FinalAlight', 'Alightings', 'TotalAlight'],
        'transfer_cols': [],
    },
    'Haifa': {
        'node_cols': ['Node', 'node'],
        'boardings_cols': ['TotalBoardings', 'Boardings', 'Boardings_Daily'],
        'alightings_cols': ['TotalAlight', 'Alightings', 'Alightings_Daily'],
        'transfer_cols': ['TransferBoardings', 'TransferAlight'],
    },
    'Haifa Metronit': {
        'node_cols': ['ModelNode', 'Node', 'node'],
        'boardings_cols': ['Boardings_Daily', 'Boardings', 'TotalBoardings'],
        'alightings_cols': ['Alightings_Daily', 'Alightings', 'TotalAlight'],
        'transfer_cols': [],
    },
    'Jerusalem': {
        'node_cols': ['ID', 'Node', 'node'],
        'boardings_cols': ['DailyBoard_2050', 'Boardings', 'Boardings_Daily'],
        'alightings_cols': ['DailyAlight_2050', 'Alightings', 'Alightings_Daily'],
        'transfer_cols': [],
    },
    'Rail': {
        'node_cols': ['Node', 'node', 'NODE_ID'],
        'boardings_cols': ['Boardings', 'TotalBoardings', 'Boardings_Daily'],
        'alightings_cols': ['Alightings', 'TotalAlight', 'Alightings_Daily'],
        'transfer_cols': [],
    },
}

# Default column config for unknown sheets
DEFAULT_COLUMN_CONFIG = {
    'node_cols': ['Node', 'node', 'NODE_ID', 'NodeID', 'ID', 'ModelNode', 'N'],
    'boardings_cols': ['Boardings', 'TotalBoardings', 'Boardings_Daily', 'InitialBoardings', 'On', 'DailyBoard_2050'],
    'alightings_cols': ['Alightings', 'TotalAlight', 'Alightings_Daily', 'FinalAlight', 'Off', 'DailyAlight_2050'],
    'transfer_cols': ['Transfers', 'TotalTransfers', 'TransferBoardings', 'TransferAlight'],
}

print("Sheet column configurations defined:")
for region in SHEET_COLUMN_CONFIG:
    print(f"  - {region}")
print(f"  - Default (for unknown sheets)")

## Step 2.5: Load Demand Data from Excel

Load demand data from Excel file with multiple regional model sheets.
Each sheet corresponds to a different transport model (Haifa, Tel Aviv, etc.).

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.5: Loading demand data from Excel...")
    
    def find_column(df_columns, possible_names):
        """Find matching column from list of possible names."""
        for name in possible_names:
            if name in df_columns:
                return name
            # Case-insensitive fallback
            for col in df_columns:
                if col.lower() == name.lower():
                    return col
        return None
    
    def process_sheet(df, sheet_name, region_name):
        """Process a single sheet and return dict of node_id -> (demand, transfers)."""
        config = SHEET_COLUMN_CONFIG.get(region_name, DEFAULT_COLUMN_CONFIG)
        
        # Find columns
        node_col = find_column(df.columns, config['node_cols'])
        boardings_col = find_column(df.columns, config['boardings_cols'])
        alightings_col = find_column(df.columns, config['alightings_cols'])
        transfer_cols = [find_column(df.columns, [tc]) for tc in config.get('transfer_cols', [])]
        transfer_cols = [tc for tc in transfer_cols if tc]
        
        if node_col is None:
            print(f"    ⚠ No node column found in '{sheet_name}'")
            return {}
        
        print(f"    Columns: node='{node_col}', boardings='{boardings_col}', alightings='{alightings_col}'")
        
        result = {}
        for _, row in df.iterrows():
            try:
                node_val = row[node_col]
                if pd.isna(node_val):
                    continue
                node_id = int(float(node_val))
                
                boardings = float(row[boardings_col]) if boardings_col and pd.notna(row.get(boardings_col)) else 0
                alightings = float(row[alightings_col]) if alightings_col and pd.notna(row.get(alightings_col)) else 0
                demand = boardings + alightings
                
                transfers = sum(float(row[tc]) for tc in transfer_cols if pd.notna(row.get(tc)))
                
                if node_id not in result:
                    result[node_id] = {'demand': demand, 'transfers': transfers}
                else:
                    result[node_id]['demand'] += demand
                    result[node_id]['transfers'] += transfers
            except (ValueError, TypeError):
                continue
        
        print(f"    ✓ Processed {len(result)} unique nodes")
        return result
    
    # Check if Excel file exists
    if not os.path.exists(DEMAND_EXCEL):
        print(f"  ✗ Demand Excel not found: {DEMAND_EXCEL}")
        print("    Update DEMAND_EXCEL path in Part 2 configuration")
        node_demand = {}
        region_demand = {}
    else:
        print(f"  Loading: {DEMAND_EXCEL}")
        xl = pd.ExcelFile(DEMAND_EXCEL)
        raw_sheet_names = xl.sheet_names
        print(f"  Found sheets: {raw_sheet_names}")
        
        node_demand = {}  # Global node -> demand mapping
        region_demand = {}  # Region -> {node -> demand} mapping
        
        for sheet in raw_sheet_names:
            # Map sheet name to region
            region_name = SHEET_NAME_MAPPING.get(sheet)
            if region_name is None:
                for key, value in SHEET_NAME_MAPPING.items():
                    if key.lower() == sheet.lower():
                        region_name = value
                        break
            
            if region_name is None:
                print(f"  ⚠ Unknown sheet '{sheet}', using default config")
                region_name = sheet
            
            try:
                df = pd.read_excel(DEMAND_EXCEL, sheet_name=sheet)
                print(f"\n  Sheet '{sheet}' → Region '{region_name}': {len(df)} rows")
                
                sheet_demand = process_sheet(df, sheet, region_name)
                
                # Store by region
                if region_name not in region_demand:
                    region_demand[region_name] = {}
                for node_id, data in sheet_demand.items():
                    if node_id not in region_demand[region_name]:
                        region_demand[region_name][node_id] = data
                    else:
                        region_demand[region_name][node_id]['demand'] += data['demand']
                        region_demand[region_name][node_id]['transfers'] += data['transfers']
                
                # Also store globally
                for node_id, data in sheet_demand.items():
                    if node_id not in node_demand:
                        node_demand[node_id] = data
                    else:
                        node_demand[node_id]['demand'] += data['demand']
                        node_demand[node_id]['transfers'] += data['transfers']
            
            except Exception as e:
                print(f"    ✗ Error: {e}")
        
        print(f"\n✓ Loaded demand for {len(node_demand)} unique nodes across all regions")
        print(f"  Regions with data: {list(region_demand.keys())}")
else:
    print("Step 2.5: Skipped (Part 2 not available)")

## Step 2.6: Match Demand to Hubs by Area

For each hub, sum demand from all its nodes using area-based matching:
1. Get hub's area (from Step 2.3)
2. Map area to region using AREA_TO_REGION
3. Look up node demand from that region's model
4. Sum demand for all nodes in the hub group

In [ ]:
import ast

if PART2_AVAILABLE and len(node_demand) > 0:
    print("Step 2.6: Matching demand to hubs by area...")
    
    matched_hubs = 0
    total_nodes_checked = 0
    total_nodes_matched = 0
    
    for idx, row in gdf_demand.iterrows():
        # Get list of nodes in this hub
        nodes_in_group = row.get('node', [])
        
        # Handle string representation of list from CSV
        if isinstance(nodes_in_group, str):
            try:
                nodes_in_group = ast.literal_eval(nodes_in_group)
            except (ValueError, SyntaxError):
                nodes_in_group = [nodes_in_group]
        
        if not isinstance(nodes_in_group, list):
            nodes_in_group = [nodes_in_group]
        
        # Get hub's area and map to region
        hub_area = row.get('area', None)
        hub_region = AREA_TO_REGION.get(hub_area) if hub_area else None
        
        # Sum demand from all nodes
        total_demand = 0
        total_transfers = 0
        nodes_matched = 0
        
        for node in nodes_in_group:
            total_nodes_checked += 1
            try:
                node = int(node)
            except (ValueError, TypeError):
                continue
            
            matched = False
            
            # Try area-based matching first
            if hub_region and hub_region in region_demand:
                if node in region_demand[hub_region]:
                    total_demand += region_demand[hub_region][node]['demand']
                    total_transfers += region_demand[hub_region][node]['transfers']
                    nodes_matched += 1
                    total_nodes_matched += 1
                    matched = True
            
            # Fall back to global matching
            if not matched and node in node_demand:
                total_demand += node_demand[node]['demand']
                total_transfers += node_demand[node]['transfers']
                nodes_matched += 1
                total_nodes_matched += 1
        
        # Assign to hub
        gdf_demand.at[idx, 'TotalDemand'] = total_demand
        gdf_demand.at[idx, 'TotalTransfers'] = total_transfers
        
        if nodes_matched > 0:
            matched_hubs += 1
    
    # Apply overlay models (Hadera, Haifa Metronit)
    overlay_regions = ['Hadera', 'Haifa Metronit']
    for overlay_region in overlay_regions:
        if overlay_region in region_demand:
            overlay_matched = 0
            for idx, row in gdf_demand.iterrows():
                nodes_in_group = row.get('node', [])
                if isinstance(nodes_in_group, str):
                    try:
                        nodes_in_group = ast.literal_eval(nodes_in_group)
                    except:
                        nodes_in_group = [nodes_in_group]
                if not isinstance(nodes_in_group, list):
                    nodes_in_group = [nodes_in_group]
                
                for node in nodes_in_group:
                    try:
                        node = int(node)
                    except:
                        continue
                    if node in region_demand[overlay_region]:
                        gdf_demand.at[idx, 'TotalDemand'] += region_demand[overlay_region][node]['demand']
                        gdf_demand.at[idx, 'TotalTransfers'] += region_demand[overlay_region][node]['transfers']
                        overlay_matched += 1
            if overlay_matched > 0:
                print(f"  Applied {overlay_region} overlay: {overlay_matched} nodes updated")
    
    # Summary
    print("\n  DEMAND MATCHING SUMMARY:")
    print(f"  ─────────────────────────")
    print(f"  Hubs processed: {len(gdf_demand)}")
    print(f"  Hubs with demand: {matched_hubs} ({matched_hubs/len(gdf_demand)*100:.1f}%)")
    print(f"  Nodes checked: {total_nodes_checked}")
    print(f"  Nodes matched: {total_nodes_matched} ({total_nodes_matched/max(total_nodes_checked,1)*100:.1f}%)")
    print(f"  Total demand: {gdf_demand['TotalDemand'].sum():,.0f}")
    print(f"  Total transfers: {gdf_demand['TotalTransfers'].sum():,.0f}")
    print(f"  Demand range: {gdf_demand['TotalDemand'].min():,.0f} - {gdf_demand['TotalDemand'].max():,.0f}")
    
    if total_nodes_matched == 0:
        print("\n  ⚠ WARNING: No nodes matched!")
        print("    Check that node IDs match between hub data and demand file")
    
    print(f"\n✓ Step 2.6 complete!")
    
elif PART2_AVAILABLE:
    print("Step 2.6: Skipped (no demand data loaded)")
else:
    print("Step 2.6: Skipped (Part 2 not available)")

## Step 2.7: Create Grouped Hubs with Demand

Aggregate individual H3 hexagons into hub groups, summing demand.

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.7: Creating grouped hubs with demand...")
    
    # Build aggregation functions
    def flatten_list(x):
        """Flatten lists and collect unique values."""
        all_items = set()
        for item in x:
            if isinstance(item, list):
                all_items.update(item)
            elif pd.notna(item):
                all_items.add(item)
        return list(all_items)
    
    def clean_nodes(x):
        """Clean and flatten node lists."""
        all_nodes = set()
        for item in x:
            if isinstance(item, list):
                for node in item:
                    if pd.notna(node):
                        try:
                            all_nodes.add(int(node))
                        except:
                            all_nodes.add(node)
            elif pd.notna(item):
                try:
                    all_nodes.add(int(item))
                except:
                    all_nodes.add(item)
        return list(all_nodes)
    
    # Build aggregation dictionary
    agg_dict = {}
    
    if 'h3_index' in gdf_demand.columns:
        agg_dict['h3_index'] = flatten_list
    if 'node' in gdf_demand.columns:
        agg_dict['node'] = clean_nodes
    if 'Mode_Planned' in gdf_demand.columns:
        agg_dict['Mode_Planned'] = flatten_list
    if 'Line_Nunique' in gdf_demand.columns:
        agg_dict['Line_Nunique'] = 'sum'
    if 'Line_Unique' in gdf_demand.columns:
        agg_dict['Line_Unique'] = flatten_list
    if 'TotalDemand' in gdf_demand.columns:
        agg_dict['TotalDemand'] = 'sum'
    if 'TotalTransfers' in gdf_demand.columns:
        agg_dict['TotalTransfers'] = 'sum'
    for col in ['address', 'area', 'district', 'metro_area']:
        if col in gdf_demand.columns:
            agg_dict[col] = 'first'
    
    # Group by 'group' column
    grouped = gdf_demand.groupby('group').agg(agg_dict).reset_index()
    
    # Calculate number of modes
    if 'Mode_Planned' in grouped.columns:
        grouped['Num_Modes'] = grouped['Mode_Planned'].apply(
            lambda x: len(x) if isinstance(x, list) else 1
        )
    
    # Dissolve geometries
    try:
        dissolved = gdf_demand.dissolve(by='group', as_index=False)
        grouped['geometry'] = dissolved['geometry'].values
    except Exception as e:
        print(f"  ⚠ Dissolve failed: {e}")
        grouped['geometry'] = gdf_demand.groupby('group')['geometry'].first().values
    
    # Create GeoDataFrame
    grouped_hubs = gpd.GeoDataFrame(grouped, geometry='geometry', crs=gdf_demand.crs)
    
    print(f"\n✓ Created {len(grouped_hubs)} hub groups")
    print(f"  Total demand: {grouped_hubs['TotalDemand'].sum():,.0f}")
    print(f"  Hubs with demand > 0: {(grouped_hubs['TotalDemand'] > 0).sum()}")
    
    # Show top hubs by demand
    top_hubs = grouped_hubs.nlargest(5, 'TotalDemand')[['group', 'TotalDemand', 'area']]
    print(f"\n  Top 5 hubs by demand:")
    print(top_hubs.to_string(index=False))
    
    print(f"\n✓ Step 2.7 complete!")
else:
    print("Step 2.7: Skipped (Part 2 not available)")

## Step 2.8: Export Grouped Hubs with Demand

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.8: Exporting grouped hubs with demand...")
    
    # Ensure output directory exists
    os.makedirs(os.path.dirname(OUTPUT_GROUPED_HUBS) if os.path.dirname(OUTPUT_GROUPED_HUBS) else '.', exist_ok=True)
    
    # Export ungrouped data with demand
    if OUTPUT_HUBS_WITH_DEMAND:
        export_ungrouped = gdf_demand.copy()
        export_ungrouped['geometry'] = export_ungrouped['geometry'].apply(lambda x: x.wkt if x else None)
        export_ungrouped.to_csv(OUTPUT_HUBS_WITH_DEMAND, index=False, encoding='utf-8-sig')
        print(f"  ✓ Ungrouped hubs: {OUTPUT_HUBS_WITH_DEMAND}")
    
    # Export grouped data
    export_grouped = grouped_hubs.copy()
    export_grouped['geometry'] = export_grouped['geometry'].apply(lambda x: x.wkt if x else None)
    export_grouped.to_csv(OUTPUT_GROUPED_HUBS, index=False, encoding='utf-8-sig')
    print(f"  ✓ Grouped hubs: {OUTPUT_GROUPED_HUBS}")
    
    print(f"\n" + "="*80)
    print("PART 2 COMPLETE - DEMAND DATA PROCESSING")
    print("="*80)
    print(f"\nResults:")
    print(f"  Hub groups: {len(grouped_hubs)}")
    print(f"  Total demand: {grouped_hubs['TotalDemand'].sum():,.0f}")
    print(f"  Hubs with demand: {(grouped_hubs['TotalDemand'] > 0).sum()}")
    print(f"\nOutput files:")
    print(f"  {OUTPUT_GROUPED_HUBS}")
    if OUTPUT_HUBS_WITH_DEMAND:
        print(f"  {OUTPUT_HUBS_WITH_DEMAND}")
    print(f"\nReady for Part 3: Influence Area Processing")
else:
    print("Step 2.8: Skipped (Part 2 not available)")

---
# PART 3: INFLUENCE AREA PROCESSING (OPTIONAL)
---

This part adds demographic data (population, employment) based on buffer zones around each hub.

**What it does:**
- Creates 3 concentric buffer zones (0-600m, 600-1000m, 1000-1200m)
- Calculates population and employment from TAZ (Traffic Analysis Zones)
- Uses proportional allocation based on area overlap
- Identifies hubs near bus terminals

**Requirements:**
- `influence_area_processor.py` module
- TAZ shapefile with `POP_2050` and `EMPL_2050` columns
- (Optional) Bus terminals shapefile

**Expected runtime:** ~2-3 minutes for 1000-2000 hubs

## Step 3.1: Configure Paths for Part 3

In [ ]:
# ============================================================================
# PART 3 CONFIGURATION - INFLUENCE AREA PROCESSING
# ============================================================================

# Input: Grouped hubs from Part 2 (or Part 1 if Part 2 was skipped)
if PART2_AVAILABLE:
    INPUT_FOR_INFLUENCE = OUTPUT_GROUPED_HUBS
else:
    INPUT_FOR_INFLUENCE = OUTPUT_H3_HEXAGONS

# Input: TAZ shapefile with population and employment
TAZ_SHAPEFILE = '/path/to/TAZ_2050.shp'

# Input: Bus terminals shapefile (optional)
BUS_TERMINALS_SHP = '/path/to/bus_terminals.shp'  # Set to None if not available

# Output: Final hubs with all data
OUTPUT_FINAL = '/path/to/output/hubs_with_complete_data.csv'
OUTPUT_FINAL_EXCEL = '/path/to/output/hubs_with_complete_data.xlsx'

# Buffer zone distances (in meters)
BUFFER_ZONES = {
    'zone1': (0, 600),      # 0-600m
    'zone2': (600, 1000),   # 600-1000m
    'zone3': (1000, 1200)   # 1000-1200m
}

print("Part 3 Configuration:")
print(f"  Input hubs: {INPUT_FOR_INFLUENCE}")
print(f"  TAZ Shapefile: {TAZ_SHAPEFILE}")
print(f"  Bus Terminals: {BUS_TERMINALS_SHP if BUS_TERMINALS_SHP else 'Not provided (optional)'}")
print(f"  Output CSV: {OUTPUT_FINAL}")
print(f"  Output Excel: {OUTPUT_FINAL_EXCEL}")
print(f"  Buffer zones: {BUFFER_ZONES}")

## Step 3.2: Load Influence Area Processor Module

In [ ]:
# Import influence area processor
# NOTE: This requires influence_area_processor.py to be in the same directory
# or in your Python path

try:
    from influence_area_processor import InfluenceAreaProcessor
    print("✓ Influence area processor module loaded successfully!")
    PART3_AVAILABLE = True
except ImportError as e:
    print("✗ Could not load influence_area_processor module")
    print(f"  Error: {e}")
    print(f"\n  To run Part 3, ensure influence_area_processor.py is available.")
    print(f"  You can continue with results from Parts 1 and 2.")
    PART3_AVAILABLE = False

## Step 3.3: Check TAZ Data Availability

In [ ]:
import os

if PART3_AVAILABLE:
    print("Checking data availability...")
    
    # Check TAZ shapefile
    if os.path.exists(TAZ_SHAPEFILE):
        print(f"  ✓ TAZ shapefile found: {TAZ_SHAPEFILE}")
        TAZ_AVAILABLE = True
    else:
        print(f"  ✗ TAZ shapefile not found: {TAZ_SHAPEFILE}")
        print(f"    Update the path in Step 3.1 or skip Part 3")
        TAZ_AVAILABLE = False
    
    # Check bus terminals (optional)
    if BUS_TERMINALS_SHP and os.path.exists(BUS_TERMINALS_SHP):
        print(f"  ✓ Bus terminals found: {BUS_TERMINALS_SHP}")
        TERMINALS_AVAILABLE = True
    elif BUS_TERMINALS_SHP:
        print(f"  ⚠ Bus terminals not found: {BUS_TERMINALS_SHP}")
        print(f"    Will proceed without terminal proximity data")
        TERMINALS_AVAILABLE = False
    else:
        print(f"  ⚠ Bus terminals not configured (optional)")
        TERMINALS_AVAILABLE = False
    
    # Check input hubs
    if os.path.exists(INPUT_FOR_INFLUENCE):
        print(f"  ✓ Input hubs found: {INPUT_FOR_INFLUENCE}")
    else:
        print(f"  ✗ Input hubs not found: {INPUT_FOR_INFLUENCE}")
        print(f"    Make sure Parts 1 and 2 completed successfully")
        TAZ_AVAILABLE = False
else:
    print("Skipping data check (influence_area_processor not available)")
    TAZ_AVAILABLE = False

## Step 3.4: Initialize Processor and Run Pipeline

In [ ]:
if PART3_AVAILABLE and TAZ_AVAILABLE:
    print("Step 3.4: Running influence area processing pipeline...")
    print("="*80)
    
    # Initialize processor
    processor = InfluenceAreaProcessor()
    
    # Set custom buffer zones if specified
    processor.buffer_zones = BUFFER_ZONES
    
    # Run full pipeline
    try:
        final_gdf = processor.process_full_pipeline(
            hubs_csv=INPUT_FOR_INFLUENCE,
            taz_shp=TAZ_SHAPEFILE,
            terminals_shp=BUS_TERMINALS_SHP if TERMINALS_AVAILABLE else None,
            output_csv=OUTPUT_FINAL
        )
        
        print(f"\n✓ Part 3 complete!")
        print(f"  Final dataset: {len(final_gdf)} hub groups")
        print(f"  Output file: {OUTPUT_FINAL}")
        
        PART3_SUCCESS = True
        
    except Exception as e:
        print(f"\n✗ Error during influence area processing:")
        print(f"  {e}")
        print(f"\n  You can still use the output from Parts 1 and 2.")
        PART3_SUCCESS = False
        final_gdf = None
else:
    if not PART3_AVAILABLE:
        print("Step 3.4: Skipped (influence_area_processor not available)")
    elif not TAZ_AVAILABLE:
        print("Step 3.4: Skipped (TAZ data not available)")
    
    print("\nParts 1 and 2 output is available:")
    if PART2_AVAILABLE:
        print(f"  {OUTPUT_GROUPED_HUBS}")
    else:
        print(f"  {OUTPUT_H3_HEXAGONS}")
    
    PART3_SUCCESS = False
    final_gdf = None

## Step 3.5: Explore Results (if Part 3 completed)

In [ ]:
if PART3_SUCCESS and final_gdf is not None:
    print("Part 3 Results Summary")
    print("="*80)
    
    # Show columns added
    influence_cols = [col for col in final_gdf.columns if 'pop_' in col or 'emp_' in col or 'terminal' in col]
    print(f"\nColumns added by influence area processing:")
    for col in influence_cols:
        print(f"  - {col}")
    
    # Statistics
    print(f"\nPopulation Statistics:")
    if 'total_pop_influence' in final_gdf.columns:
        print(f"  Total population in influence areas: {final_gdf['total_pop_influence'].sum():,.0f}")
        print(f"  Average per hub: {final_gdf['total_pop_influence'].mean():,.0f}")
        print(f"  Max in single hub: {final_gdf['total_pop_influence'].max():,.0f}")
    
    print(f"\nEmployment Statistics:")
    if 'total_emp_influence' in final_gdf.columns:
        print(f"  Total employment in influence areas: {final_gdf['total_emp_influence'].sum():,.0f}")
        print(f"  Average per hub: {final_gdf['total_emp_influence'].mean():,.0f}")
        print(f"  Max in single hub: {final_gdf['total_emp_influence'].max():,.0f}")
    
    # Bus terminal proximity
    if 'near_bus_terminal' in final_gdf.columns:
        near_terminals = final_gdf['near_bus_terminal'].sum()
        print(f"\nBus Terminal Proximity:")
        print(f"  Hubs within 200m of terminal: {near_terminals}")
        print(f"  Percentage: {near_terminals/len(final_gdf)*100:.1f}%")
    
    # Preview data
    print(f"\nFirst 3 records:")
    display_cols = ['group', 'total_pop_influence', 'total_emp_influence']
    if 'TotalDemand' in final_gdf.columns:
        display_cols.append('TotalDemand')
    if 'near_bus_terminal' in final_gdf.columns:
        display_cols.append('near_bus_terminal')
    
    print(final_gdf[display_cols].head(3))
else:
    print("Part 3 did not complete - no results to display")

## Step 3.6: Export to Excel with Multiple Sheets (Optional)

In [ ]:
if PART3_SUCCESS and final_gdf is not None:
    print("Exporting to Excel with analysis sheets...")
    
    try:
        with pd.ExcelWriter(OUTPUT_FINAL_EXCEL, engine='openpyxl') as writer:
            # Main data sheet
            export_df = final_gdf.copy()
            if 'geometry' in export_df.columns:
                export_df['geometry'] = export_df['geometry'].apply(lambda x: x.wkt if x else None)
            export_df.to_excel(writer, sheet_name='Hub Groups', index=False)
            
            # Top hubs by population
            if 'total_pop_influence' in final_gdf.columns:
                top_pop = final_gdf.nlargest(20, 'total_pop_influence')[[
                    'group', 'total_pop_influence', 'total_emp_influence', 'address'
                ]].copy()
                top_pop.to_excel(writer, sheet_name='Top by Population', index=False)
            
            # Top hubs by employment
            if 'total_emp_influence' in final_gdf.columns:
                top_emp = final_gdf.nlargest(20, 'total_emp_influence')[[
                    'group', 'total_pop_influence', 'total_emp_influence', 'address'
                ]].copy()
                top_emp.to_excel(writer, sheet_name='Top by Employment', index=False)
            
            # Combined score (if demand data available)
            if 'TotalDemand' in final_gdf.columns and 'total_pop_influence' in final_gdf.columns:
                combined = final_gdf.copy()
                # Normalize both metrics to 0-1 scale
                combined['demand_norm'] = (combined['TotalDemand'] - combined['TotalDemand'].min()) / \
                                          (combined['TotalDemand'].max() - combined['TotalDemand'].min())
                combined['pop_norm'] = (combined['total_pop_influence'] - combined['total_pop_influence'].min()) / \
                                       (combined['total_pop_influence'].max() - combined['total_pop_influence'].min())
                combined['combined_score'] = (combined['demand_norm'] + combined['pop_norm']) / 2
                
                top_combined = combined.nlargest(20, 'combined_score')[[
                    'group', 'TotalDemand', 'total_pop_influence', 'combined_score', 'address'
                ]].copy()
                top_combined.to_excel(writer, sheet_name='Top by Combined Score', index=False)
            
            # Zone statistics
            zone_stats = pd.DataFrame({
                'Zone': ['Zone 1 (0-600m)', 'Zone 2 (600-1000m)', 'Zone 3 (1000-1200m)', 'Total'],
                'Avg Population': [
                    final_gdf['pop_zone1'].mean() if 'pop_zone1' in final_gdf.columns else 0,
                    final_gdf['pop_zone2'].mean() if 'pop_zone2' in final_gdf.columns else 0,
                    final_gdf['pop_zone3'].mean() if 'pop_zone3' in final_gdf.columns else 0,
                    final_gdf['total_pop_influence'].mean() if 'total_pop_influence' in final_gdf.columns else 0
                ],
                'Avg Employment': [
                    final_gdf['emp_zone1'].mean() if 'emp_zone1' in final_gdf.columns else 0,
                    final_gdf['emp_zone2'].mean() if 'emp_zone2' in final_gdf.columns else 0,
                    final_gdf['emp_zone3'].mean() if 'emp_zone3' in final_gdf.columns else 0,
                    final_gdf['total_emp_influence'].mean() if 'total_emp_influence' in final_gdf.columns else 0
                ]
            })
            zone_stats.to_excel(writer, sheet_name='Zone Statistics', index=False)
        
        print(f"✓ Excel file created: {OUTPUT_FINAL_EXCEL}")
        print(f"  Sheets: Hub Groups, Top by Population, Top by Employment, Zone Statistics")
        if 'TotalDemand' in final_gdf.columns:
            print(f"          Top by Combined Score")
    
    except Exception as e:
        print(f"⚠ Could not create Excel file: {e}")
        print(f"  CSV file is still available: {OUTPUT_FINAL}")
else:
    print("Skipping Excel export (Part 3 did not complete)")

---
# PART 4: SCORING AND PRIORITIZATION
---

This part implements the hub prioritization scoring system:

**What it does:**
1. Data cleaning (convert string lists to proper lists)
2. Mode name standardization
3. Regional and location categorization
4. Mode service score calculation
5. Bus terminal scoring
6. Hub type classification (National/Regional/Local/Train Station)
7. 5 criteria scoring (all normalized 1-10 per hub type):
   - RegionLocation (geographic/strategic importance)
   - bus_terminal (bus integration)
   - score (mode service diversity)
   - TotalDemand (log-transformed)
   - PopEmp (population/employment with distance decay)
8. Monte Carlo aggregation (10,000 iterations with random weights)
9. Rankings (overall and within hub type)

**Based on:** Group_n_Filter_Hubs.ipynb and HubsScoring_vAugust2025.ipynb

**Expected runtime:** ~30 seconds for 100-200 hubs

In [ ]:
print("="*80)
print("COMPLETE PIPELINE SUMMARY")
print("="*80)

print(f"\nPart 1: H3 Hexagon Processing")
print(f"  ✓ Created {len(gdf_output_part1)} H3 hexagons")
print(f"  ✓ Identified {gdf_output_part1['group'].nunique()} proximity groups")
print(f"  ✓ Output: {OUTPUT_H3_HEXAGONS}")

if PART2_AVAILABLE:
    print(f"\nPart 2: Demand Data Processing")
    print(f"  ✓ Added daily demand data")
    print(f"  ✓ Created {len(grouped_hubs)} hub groups")
    print(f"  ✓ Total demand: {grouped_hubs['TotalDemand'].sum():,.0f}")
    print(f"  ✓ Output: {OUTPUT_GROUPED_HUBS}")
else:
    print(f"\nPart 2: Demand Data Processing")
    print(f"  ✗ Skipped")

print(f"\nPart 3: Influence Area Processing")
if PART3_SUCCESS:
    print(f"  ✓ Added population and employment data")
    print(f"  ✓ Created buffer zone statistics")
    if TERMINALS_AVAILABLE:
        print(f"  ✓ Identified terminal proximity")
    print(f"  ✓ Output CSV: {OUTPUT_FINAL}")
    print(f"  ✓ Output Excel: {OUTPUT_FINAL_EXCEL}")
elif PART3_AVAILABLE and not TAZ_AVAILABLE:
    print(f"  ✗ Skipped (TAZ data not available)")
else:
    print(f"  ✗ Skipped")

print(f"\nPart 4: Scoring and Prioritization")
if 'df_scored' in locals() and df_scored is not None:
    print(f"  ✓ Scored and ranked {len(df_scored)} hubs")
    print(f"  ✓ Hub types: {df_scored['HubType'].value_counts().to_dict()}")
    print(f"  ✓ Score range: {df_scored['Average_Simulated_Score'].min():.3f} - {df_scored['Average_Simulated_Score'].max():.3f}")
    print(f"  ✓ Monte Carlo: {MONTE_CARLO_ITERATIONS:,} iterations")
    print(f"  ✓ Output CSV: {OUTPUT_SCORED_HUBS}")
    print(f"  ✓ Output Excel: {OUTPUT_SCORED_EXCEL}")
else:
    print(f"  ✗ Not run")

print(f"\n" + "="*80)
print(f"PIPELINE COMPLETE!")
print("="*80)

# Summary of available outputs
print(f"\nAvailable Output Files:")
print(f"  1. H3 Hexagons: {OUTPUT_H3_HEXAGONS}")
if PART2_AVAILABLE:
    print(f"  2. With Demand: {OUTPUT_GROUPED_HUBS}")
if PART3_SUCCESS:
    print(f"  3. Complete Data (CSV): {OUTPUT_FINAL}")
    print(f"  4. Complete Data (Excel): {OUTPUT_FINAL_EXCEL}")
if 'df_scored' in locals() and df_scored is not None:
    print(f"  5. Scored Hubs (CSV): {OUTPUT_SCORED_HUBS}")
    print(f"  6. Scored Hubs (Excel): {OUTPUT_SCORED_EXCEL}")

print(f"\n" + "="*80)
print("Ready for analysis and visualization!")
print("="*80)

## Step 4.7: Export Scored and Ranked Hubs

In [ ]:
# Set defaults if not already configuredif 'MONTE_CARLO_ITERATIONS' not in globals():    MONTE_CARLO_ITERATIONS = 10000if 'RANDOM_SEED' not in globals():    RANDOM_SEED = 42if 'TIER_NATIONAL' not in globals():    TIER_NATIONAL = "ארצי"if 'TIER_METRO' not in globals():    TIER_METRO = "מטרופוליני"if 'TIER_LOCAL' not in globals():    TIER_LOCAL = "עירוני"print(f"Step 4.6: Running Monte Carlo simulation ({MONTE_CARLO_ITERATIONS:,} iterations)...")scoring_cols = ['RegionLocation_Norm', 'bus_terminal_Norm', 'score_Norm',                 'TotalDemand_Norm', 'PopEmp_Score_Norm']# Check all columns existmissing = [col for col in scoring_cols if col not in df_scoring.columns]if missing:    print(f"  ⚠ Missing score columns: {missing}")    print(f"  Skipping Monte Carlo")    df_scoring['Average_Simulated_Score'] = 5.0else:    np.random.seed(RANDOM_SEED)    results = []        for hub_type in df_scoring['HubType'].unique():        hub_df = df_scoring[df_scoring['HubType'] == hub_type].copy()        hub_df['Sum_Simulated_Scores'] = 0.0                for i in range(MONTE_CARLO_ITERATIONS):            # Generate random weights (max 0.5 per criterion)            weights = np.random.rand(len(scoring_cols))            while any(weights > 0.5):                weights = np.random.rand(len(scoring_cols))            weights /= weights.sum()                        # Calculate weighted scores            hub_df['Sum_Simulated_Scores'] += (hub_df[scoring_cols] * weights).sum(axis=1)                # Average across iterations        hub_df['Average_Simulated_Score'] = hub_df['Sum_Simulated_Scores'] / MONTE_CARLO_ITERATIONS        hub_df = hub_df.drop(columns='Sum_Simulated_Scores')        results.append(hub_df)        df_scored = pd.concat(results, ignore_index=False).sort_index()        # Add rankings    df_scored['Rank_within_HubType'] = df_scored.groupby('HubType')['Average_Simulated_Score'].rank(        method='dense', ascending=False    )    df_scored['Overall_Rank'] = df_scored['Average_Simulated_Score'].rank(        method='dense', ascending=False    )        print(f"\n✓ Monte Carlo complete!")    print(f"  Final scores: {df_scored['Average_Simulated_Score'].min():.3f} - {df_scored['Average_Simulated_Score'].max():.3f}")    print(f"  Mean: {df_scored['Average_Simulated_Score'].mean():.3f}")        print(f"\n  Top 10 hubs overall:")    top_10 = df_scored.nlargest(10, 'Average_Simulated_Score')    for i, (idx, row) in enumerate(top_10.iterrows(), 1):        hub_id = row.get('group', idx)        score = row['Average_Simulated_Score']        hub_type = row['HubType']        demand = row.get('TotalDemand', 0)        print(f"    {i}. Hub {hub_id} ({hub_type}): {score:.3f} | Demand: {demand:,.0f}")        # Summary by hub type    print(f"\n  Results by Hub Type:")    for hub_type in df_scored['HubType'].unique():        hub_data = df_scored[df_scored['HubType'] == hub_type]        print(f"    {hub_type}: {len(hub_data)} hubs, scores {hub_data['Average_Simulated_Score'].min():.3f}-{hub_data['Average_Simulated_Score'].max():.3f}")

## Step 4.6: Monte Carlo Aggregation and Rankings

In [ ]:
# Set distance decay default if not configuredif 'DISTANCE_DECAY_BETA' not in globals():    DISTANCE_DECAY_BETA = 1.5print("Step 4.5: Normalizing scores and calculating Pop/Emp score...")# Normalization functiondef normalize_col_by_type(df, col, hub_type_col='HubType'):    """Normalize to 1-10 scale per hub type."""    normalized_col = col + '_Norm'    for hub_type in df[hub_type_col].unique():        mask = df[hub_type_col] == hub_type        values = df.loc[mask, col]        min_val, max_val = values.min(), values.max()        if max_val > min_val:            df.loc[mask, normalized_col] = 1 + (values - min_val) * 9 / (max_val - min_val)        else:            df.loc[mask, normalized_col] = 5.5    return df# Normalize categorical scoresfor col in ['RegionLocation', 'score', 'bus_terminal']:    df_scoring = normalize_col_by_type(df_scoring, col)# Normalize demand with log10if 'TotalDemand' in df_scoring.columns:    df_scoring['LogDemand'] = df_scoring['TotalDemand'].apply(lambda x: 0 if x == 0 else np.log10(x))    for hub_type in df_scoring['HubType'].unique():        mask = df_scoring['HubType'] == hub_type        values = df_scoring.loc[mask, 'LogDemand']        values_nonzero = values[values > 0]        if len(values_nonzero) > 0:            min_val, max_val = values_nonzero.min(), values_nonzero.max()            if max_val > min_val:                df_scoring.loc[mask, 'TotalDemand_Norm'] = 1 + (values - min_val) * 9 / (max_val - min_val)            else:                df_scoring.loc[mask, 'TotalDemand_Norm'] = 5.5        else:            df_scoring.loc[mask, 'TotalDemand_Norm'] = 1.0# Pop/Emp score with distance decaypop_cols = ['pop_0_500', 'pop_500_1000', 'pop_1000_1500']emp_cols = ['emp_0_500', 'emp_500_1000', 'emp_1000_1500']if all(col in df_scoring.columns for col in pop_cols + emp_cols):    radii = [0, 500, 1000, 1500]    mids = np.array([(radii[i] + radii[i+1]) / 2 for i in range(len(radii)-1)])    decay = mids ** DISTANCE_DECAY_BETA        scores = []    for _, row in df_scoring.iterrows():        pop = np.array([row.get(col, 0) for col in pop_cols], dtype=float)        emp = np.array([row.get(col, 0) for col in emp_cols], dtype=float)                # Weight by hub type        if row['HubType'] in ['National', 'Regional', TIER_NATIONAL, TIER_METRO]:            w_pop, w_emp = 0.2, 0.8        else:            w_pop, w_emp = 0.8, 0.2                combined = w_pop * pop + w_emp * emp        score = (combined / decay).sum()        scores.append(score)        df_scoring['PopEmp_Score_Raw'] = scores        # Normalize per hub type    for hub_type in df_scoring['HubType'].unique():        mask = df_scoring['HubType'] == hub_type        values = df_scoring.loc[mask, 'PopEmp_Score_Raw']        min_val, max_val = values.min(), values.max()        if max_val > min_val:            df_scoring.loc[mask, 'PopEmp_Score_Norm'] = 1 + (values - min_val) * 9 / (max_val - min_val)        else:            df_scoring.loc[mask, 'PopEmp_Score_Norm'] = 5.5else:    print("  ⚠ Pop/Emp columns not found, using default score")    df_scoring['PopEmp_Score_Norm'] = 5.0print(f"✓ Scores normalized")for col in ['RegionLocation_Norm', 'score_Norm', 'bus_terminal_Norm', 'TotalDemand_Norm', 'PopEmp_Score_Norm']:    if col in df_scoring.columns:        print(f"  {col}: {df_scoring[col].min():.2f} - {df_scoring[col].max():.2f}")

## Step 4.5: Normalize Scores and Calculate Pop/Emp Score

In [ ]:
# Set tier defaults if not configuredif 'TIER_NATIONAL' not in globals():    TIER_NATIONAL = "ארצי"if 'TIER_METRO' not in globals():    TIER_METRO = "מטרופוליני"if 'TIER_LOCAL' not in globals():    TIER_LOCAL = "עירוני"print("Step 4.4: Filtering eligible hubs and classifying types...")# Filter: >= 1000 demand AND >= 2 modesinitial_count = len(df_scoring)if 'TotalDemand' in df_scoring.columns:    df_scoring = df_scoring[df_scoring['TotalDemand'] >= 1000].copy()    print(f"  After demand filter (>=1000): {len(df_scoring)} hubs")df_scoring = df_scoring[df_scoring['Mode_Planned'].apply(lambda x: len(x) > 1)].copy()print(f"  After mode filter (>=2 modes): {len(df_scoring)} hubs")print(f"  Filtered out: {initial_count - len(df_scoring)} hubs")# Hub type classificationdef classify_hub_tier(total_demand, modes, num_lines):    """Classify hub based on modes, lines, and ridership."""    if not isinstance(modes, list):        modes = [modes] if pd.notna(modes) else []        has_highspeed = 'HighSpeed Rail' in modes or 'HighSpeed' in modes    has_interurban = 'Interurban Rail' in modes or 'Interurban' in modes    has_suburban = 'Suburban Rail' in modes or 'Suburban' in modes    has_metro = 'Metro' in modes    has_brt = 'BRT' in modes    has_lrt = 'LRT' in modes    has_rail = 'Rail' in modes        # National: (HighSpeed OR Interurban) AND >=3 lines AND >=50k demand    if (has_highspeed or has_interurban or (has_rail and num_lines >= 3)) and (num_lines >= 3) and (total_demand >= 50000):        return TIER_NATIONAL    # Regional: (Suburban OR Metro OR Interurban OR HighSpeed) AND >=3 lines    elif (has_suburban or has_metro or has_interurban or has_highspeed or has_rail) and (num_lines >= 3):        return TIER_METRO    # Local: (BRT OR LRT) AND >=3 lines AND >=1000 demand    elif (has_brt or has_lrt) and (num_lines >= 3) and (total_demand >= 1000):        return TIER_LOCAL    # Train Station: rail modes but <=2 lines    elif (has_interurban or has_highspeed or has_suburban or has_rail) and (num_lines <= 2):        return 'Train Station'    else:        return 'Not Hub'df_scoring['HubType'] = df_scoring.apply(    lambda row: classify_hub_tier(        total_demand=row.get('TotalDemand', 0),        modes=row['Mode_Planned'],        num_lines=row['Total_Unique_Lines']    ),    axis=1)print(f"\n✓ Hub types classified:")print(df_scoring['HubType'].value_counts().to_string())

## Step 4.4: Filter Eligible Hubs and Classify Hub Types

In [ ]:
# Set mode weights default if not configuredif 'MODE_WEIGHTS' not in globals():    MODE_WEIGHTS = {        'Funicular': 1.0,        'Cable Line': 2.0,        'BRT': 3.0,        'LRT': 4.0,        'Metro': 5.0,        'Suburban Rail': 6.0,        'Interurban Rail': 7.0,        'HighSpeed Rail': 8.0,        'Rail': 7.0,        'Express Bus': 3.0,        'Bus': 2.0,    }print("Step 4.3: Calculating mode service score and bus terminal score...")# Count modes with linesdef count_positive_mode_lines(row):    """Count how many modes have lines."""    count = 0    mode_line_cols = ['BRT Lines', 'Cable Line Lines', 'Funicular Lines',                      'HighSpeed Rail Lines', 'Interurban Rail Lines', 'LRT Lines',                      'Metro Lines', 'Suburban Rail Lines']    for col in mode_line_cols:        if col in row.index and row[col] > 0:            count += 1    return count# Mode service scoredef get_mode_score(row):    """Calculate mode service score with diversity bonus."""    score = 0.0    alpha = 0.1  # Diversity bonus factor        for mode, weight in MODE_WEIGHTS.items():        column_name = f'{mode} Lines'        if column_name in row.index and row[column_name] > 0:            score += row[column_name] * weight        # Apply diversity bonus    n_modes = row.get('Num_Modes', 1)    score = score * (1 + alpha * (n_modes - 1))        return scoredf_scoring['Num_Modes'] = df_scoring.apply(count_positive_mode_lines, axis=1)df_scoring['score'] = df_scoring.apply(get_mode_score, axis=1)# Bus terminal scoredef bus_terminal_score(term_type):    """Convert terminal type to score."""    if pd.isna(term_type):        return 0    term_type = str(term_type).strip()    if 'חניון לילה' in term_type:        return 1    elif 'מסוף קטן' in term_type or 'מסוף בינוני' in term_type:        return 2    elif 'מסוף גדול' in term_type or 'מתקן משולב' in term_type:        return 3    return 0if 'term_type' in df_scoring.columns:    df_scoring['bus_terminal'] = df_scoring['term_type'].apply(bus_terminal_score)else:    df_scoring['bus_terminal'] = 0print(f"✓ Mode scores calculated")print(f"  Score range: {df_scoring['score'].min():.2f} - {df_scoring['score'].max():.2f}")print(f"  Bus terminal score dist: {df_scoring['bus_terminal'].value_counts().to_dict()}")

## Step 4.3: Calculate Mode Score and Bus Terminal Score

In [ ]:
print("Step 4.2: Data cleaning and preparation...")

# Make a working copy
df_scoring = INPUT_FOR_SCORING.copy()

# Convert string representations to lists
def split_list_string(txt):
    """Convert string representation of list to actual list."""
    if pd.isna(txt) or txt == '':
        return []
    if isinstance(txt, list):
        return txt
    txt = str(txt).replace("'", "").replace("[", "").replace("]", "").replace('"', "")
    return [x.strip() for x in txt.split(',') if x.strip()]

# Fix Mode_Planned column
if 'Mode_Planned' in df_scoring.columns:
    df_scoring['Mode_Planned'] = df_scoring['Mode_Planned'].apply(split_list_string)
    
# Fix Line_Unique column
if 'Line_Unique' in df_scoring.columns:
    df_scoring['Line_Unique'] = df_scoring['Line_Unique'].apply(split_list_string)

# Fix mode names
def correct_mode_planned(modes):
    """Clean up mode names and standardize."""
    if not isinstance(modes, list):
        return []
    cleaned = []
    for mode in modes:
        mode = str(mode).strip().replace(',', '')
        if mode == 'HighSpeed':
            cleaned.append('HighSpeed Rail')
        elif mode == 'Interurban':
            cleaned.append('Interurban Rail')
        elif mode == 'Suburban':
            cleaned.append('Suburban Rail')
        elif mode != 'Rail':
            cleaned.append(mode)
    # Remove generic 'Rail' if we have specific types
    if 'Rail' in cleaned:
        specific_rails = ['HighSpeed Rail', 'Interurban Rail', 'Suburban Rail']
        if any(r in cleaned for r in specific_rails):
            cleaned = [m for m in cleaned if m != 'Rail']
    return cleaned

df_scoring['Mode_Planned'] = df_scoring['Mode_Planned'].apply(correct_mode_planned)

# Count unique lines
if 'Total_Unique_Lines' not in df_scoring.columns:
    df_scoring['Total_Unique_Lines'] = df_scoring['Line_Unique'].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

# Add region/location categories
def get_region_category(area):
    """0 for Tel Aviv, 1 for others."""
    if pd.isna(area):
        return 0
    area = str(area).strip()
    if 'תל' in area or 'Tel Aviv' in area:
        return 0
    return 1

def get_location_category(location):
    """3 for Core (גלעין), 2 for Ring (טבעת), 1 for periphery."""
    if pd.isna(location):
        return 1
    location = str(location).strip()
    if 'גלעין' in location:
        return 3
    elif 'טבעת' in location:
        return 2
    else:
        return 1

if 'area' in df_scoring.columns:
    df_scoring['Region_category'] = df_scoring['area'].apply(get_region_category)
else:
    df_scoring['Region_category'] = 1

if 'location' in df_scoring.columns:
    df_scoring['Location_category'] = df_scoring['location'].apply(get_location_category)
else:
    df_scoring['Location_category'] = 1

df_scoring['RegionLocation'] = df_scoring['Region_category'] * df_scoring['Location_category']

print(f"✓ Data cleaned and prepared")
print(f"  Rows: {len(df_scoring)}")
print(f"  Unique modes: {set([m for modes in df_scoring['Mode_Planned'] for m in modes])}")
print(f"  Region categories: {df_scoring['Region_category'].value_counts().to_dict()}")

## Step 4.2: Data Cleaning and Preparation

In [ ]:
# ============================================================================
# PART 4 CONFIGURATION - SCORING AND PRIORITIZATION
# ============================================================================

# Input: Use output from Part 2 or Part 3 if available
if PART3_SUCCESS and final_gdf is not None:
    INPUT_FOR_SCORING = final_gdf.copy()
    print("Using Part 3 output (with influence area data)")
elif PART2_AVAILABLE:
    INPUT_FOR_SCORING = grouped_hubs.copy()
    print("Using Part 2 output (with demand data)")
else:
    INPUT_FOR_SCORING = gdf_h3.copy()
    print("Using Part 1 output (H3 hexagons only)")

# Output: Final scored and ranked hubs
OUTPUT_SCORED_HUBS = '/path/to/output/scored_hubs_final.csv'
OUTPUT_SCORED_EXCEL = '/path/to/output/scored_hubs_final.xlsx'

# Scoring configuration
MONTE_CARLO_ITERATIONS = 10000
RANDOM_SEED = 42
DISTANCE_DECAY_BETA = 1.5

# Tier names
TIER_NATIONAL = "ארצי"
TIER_METRO = "מטרופוליני"  
TIER_LOCAL = "עירוני"

# Mode weights (from Group_n_Filter_Hubs notebook)
MODE_WEIGHTS = {
    'Funicular': 1.0,
    'Cable Line': 2.0,
    'BRT': 3.0,
    'LRT': 4.0,
    'Metro': 5.0,
    'Suburban Rail': 6.0,
    'Interurban Rail': 7.0,
    'HighSpeed Rail': 8.0,
    'Rail': 7.0,
    'Express Bus': 3.0,
    'Bus': 2.0,
}

print(f"\nPart 4 Configuration:")
print(f"  Input: {len(INPUT_FOR_SCORING)} hub groups")
print(f"  Monte Carlo iterations: {MONTE_CARLO_ITERATIONS:,}")
print(f"  Distance decay beta: {DISTANCE_DECAY_BETA}")
print(f"  Output CSV: {OUTPUT_SCORED_HUBS}")
print(f"  Output Excel: {OUTPUT_SCORED_EXCEL}")

## Step 4.1: Configuration for Scoring

In [ ]:
print("="*80)
print("PIPELINE COMPLETE SUMMARY")
print("="*80)

print(f"\nPart 1: H3 Hexagon Processing")
print(f"  ✓ Created {len(gdf_output_part1)} H3 hexagons")
print(f"  ✓ Identified {gdf_output_part1['group'].nunique()} proximity groups")
print(f"  ✓ Output: {OUTPUT_H3_HEXAGONS}")

if PART2_AVAILABLE:
    print(f"\nPart 2: Demand Data Processing")
    print(f"  ✓ Added daily demand data")
    print(f"  ✓ Output: {OUTPUT_GROUPED_HUBS}")
else:
    print(f"\nPart 2: Demand Data Processing")
    print(f"  ✗ Skipped (module not available)")

print(f"\nPart 3: Influence Area Processing")
if PART3_SUCCESS:
    print(f"  ✓ Added population and employment data")
    print(f"  ✓ Created buffer zone statistics")
    if TERMINALS_AVAILABLE:
        print(f"  ✓ Identified terminal proximity")
    print(f"  ✓ Output CSV: {OUTPUT_FINAL}")
    print(f"  ✓ Output Excel: {OUTPUT_FINAL_EXCEL}")
elif PART3_AVAILABLE and not TAZ_AVAILABLE:
    print(f"  ✗ Skipped (TAZ data not available)")
elif not PART3_AVAILABLE:
    print(f"  ✗ Skipped (module not available)")
else:
    print(f"  ✗ Error during processing")

print(f"\n" + "="*80)
print(f"Pipeline complete! Check output files above.")
print("="*80)

# Summary of available outputs
print(f"\nAvailable Output Files:")
print(f"  1. H3 Hexagons: {OUTPUT_H3_HEXAGONS}")
if PART2_AVAILABLE:
    print(f"  2. With Demand: {OUTPUT_GROUPED_HUBS}")
if PART3_SUCCESS:
    print(f"  3. Complete Data (CSV): {OUTPUT_FINAL}")
    print(f"  4. Complete Data (Excel): {OUTPUT_FINAL_EXCEL}")